<a href="https://colab.research.google.com/github/AKChumba/NLP_LABS/blob/main/NAT820S_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAT820S — Practical Lab 1: Corpora, Pre-Processing & Feature Extraction

**Weeks 3–5 | Natural Language Processing and Applications**

In this lab you'll get hands-on with everything from Lecture 3:
- Building a text corpus
- Tokenization
- Stemming & lemmatization
- One-Hot Encoding, Bag of Words, Bag of N-Grams
- TF-IDF
- Word embeddings with Word2Vec

Look out for **🔧 TODO** cells — that's where you write code. Everything else is a worked example you can run and adapt.

> Reference: Gupta, Majumder & Vajjala (2020), *Practical Natural Language Processing*, O'Reilly.

**How to use this notebook:** Run each cell in order (Shift+Enter). Read the markdown before each code cell — it tells you what the code does and what you need to do next.

## Part 0 — Setup

Run this first. It installs/loads everything we need for the whole lab.

In [ ]:
# Core imports
import nltk
import pandas as pd
import numpy as np

# Download the NLTK data we need (only runs once per Colab session)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Setup complete ✅")

Setup complete ✅


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Part 1 — Building a Text Corpus

Recall from the lecture: a **corpus** is a structured collection of text documents. We'll reuse the same toy corpus from class throughout this lab so you can compare your code's output to what we discussed.

In [ ]:
# Our toy corpus (same one from the lecture)
corpus = [
    "Dog bites man.",
    "Man bites dog.",
    "Dog eats meat.",
    "Man eats food."
]

for i, doc in enumerate(corpus, start=1):
    print(f"D{i}: {doc}")

D1: Dog bites man.
D2: Man bites dog.
D3: Dog eats meat.
D4: Man eats food.


### 🔧 TODO 1.1
Add **two more sentences** to the corpus below, reusing some of the same words (`dog`, `man`, `bites`, `eats`, `meat`, `food`) plus at least one new word. Store the result in a new list called `my_corpus`.

In [ ]:
# TODO: create my_corpus as a copy of `corpus` plus your 2 new sentences
my_corpus = corpus.copy()
my_corpus.append("Dog eat zebra")
my_corpus.append("Man eats food")

my_corpus

['Dog bites man.',
 'Man bites dog.',
 'Dog eats meat.',
 'Man eats food.',
 'Dog eat zebra',
 'Man eats food']

## Part 2 — Tokenization

Tokenization splits raw text into sentences, then into words. NLTK's `sent_tokenize` and `word_tokenize` handle most of the heavy lifting.

In [ ]:
paragraph = (
    "In the previous lecture, we saw a quick overview of what is NLP, what are some "
    "of the common applications and challenges in NLP. This lecture, we will learn "
    "about the various pre-processing steps and how they play important roles in "
    "solving NLP problems."
)

sentences = sent_tokenize(paragraph)
for s in sentences:
    print("SENTENCE:", s)
    print("TOKENS:  ", word_tokenize(s))
    print()

SENTENCE: In the previous lecture, we saw a quick overview of what is NLP, what are some of the common applications and challenges in NLP.
TOKENS:   ['In', 'the', 'previous', 'lecture', ',', 'we', 'saw', 'a', 'quick', 'overview', 'of', 'what', 'is', 'NLP', ',', 'what', 'are', 'some', 'of', 'the', 'common', 'applications', 'and', 'challenges', 'in', 'NLP', '.']

SENTENCE: This lecture, we will learn about the various pre-processing steps and how they play important roles in solving NLP problems.
TOKENS:   ['This', 'lecture', ',', 'we', 'will', 'learn', 'about', 'the', 'various', 'pre-processing', 'steps', 'and', 'how', 'they', 'play', 'important', 'roles', 'in', 'solving', 'NLP', 'problems', '.']



### 🔧 TODO 2.1
Tokenize **every document** in `corpus` into words. Print the token list for each document.

In [ ]:
# TODO: loop over `corpus`, tokenize each document with word_tokenize, and print the tokens
for doc in corpus:
    pass  # replace this with your code

### 🔧 TODO 2.2 — Reflection
Run the cell below on a trickier sentence, then answer in the markdown cell underneath: **What goes wrong, and why?** (Hint: look closely at the punctuation.)

In [ ]:
tricky_sentence = "Mr. Jack O'Neil works at Melitas Marg, located at 245 Yonge Avenue, Austin, 70272."
print(word_tokenize(tricky_sentence))

**Your answer here:**

*(double-click to edit this cell)*


## Part 3 — Stemming & Lemmatization

- **Stemming** chops suffixes off using fixed rules (fast, sometimes linguistically wrong).
- **Lemmatization** maps a word to its dictionary base form using linguistic knowledge (slower, more accurate).

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words_to_try = ["cars", "revolution", "better", "meeting", "ate"]

for w in words_to_try:
    print(f"{w:12s} -> stem: {stemmer.stem(w):10s} | lemma: {lemmatizer.lemmatize(w, pos='v')}")

### 🔧 TODO 3.1
Build a small `pandas` DataFrame with three columns — `word`, `stem`, `lemma` — for **every unique word** in `corpus` (lowercased, punctuation removed). Some starter code is provided.

In [ ]:
import string

# collect unique lowercase words from the corpus, punctuation stripped
unique_words = set()
for doc in corpus:
    for tok in word_tokenize(doc.lower()):
        if tok not in string.punctuation:
            unique_words.add(tok)

# TODO: for each word in unique_words, compute its stem and lemma,
# then build a DataFrame with columns: word, stem, lemma
rows = []
# for w in sorted(unique_words):
#     rows.append({"word": w, "stem": ..., "lemma": ...})

pd.DataFrame(rows)

### 🔧 TODO 3.2 — Reflection
Look at your table. Is there a word where stemming and lemmatization give **different, non-obvious** results? Which would you trust more, and why?

**Your answer here:**


## Part 4 — One-Hot Encoding

Recall the scheme from the lecture: `dog=1, bites=2, man=3, meat=4, food=5, eats=6`. Each word becomes a 6-dimensional binary vector.

In [ ]:
vocab = {"dog": 1, "bites": 2, "man": 3, "meat": 4, "food": 5, "eats": 6}

def get_onehot_vector(sentence, vocab):
    onehot_encoded = []
    for word in sentence.lower().replace(".", "").split():
        temp = [0] * len(vocab)
        if word in vocab:
            temp[vocab[word] - 1] = 1
        onehot_encoded.append(temp)
    return onehot_encoded

get_onehot_vector("dog bites man", vocab)

### 🔧 TODO 4.1
Compute the one-hot encoding for **D4** (`"Man eats food."`) and for a **new sentence** `"man eats fruits"`. What happens with the word "fruits"?

In [ ]:
# TODO: call get_onehot_vector on D4 and on "man eats fruits"


### 🔧 TODO 4.2 — Reflection
Why can't one-hot encoding represent the word *"fruits"*? What would you have to do to fix this? (This is the **out-of-vocabulary / OOV problem** from the lecture.)

**Your answer here:**


## Part 5 — Bag of Words & Bag of N-Grams

`CountVectorizer` from scikit-learn builds a Bag-of-Words representation automatically.

In [ ]:
count_vect = CountVectorizer()
bow_rep = count_vect.fit_transform(corpus)

print("Vocabulary:", count_vect.vocabulary_)
print()
print("BoW for D1 ('Dog bites man.'):", bow_rep[0].toarray())
print("BoW for D2 ('Man bites dog.'):", bow_rep[1].toarray())

new_doc = count_vect.transform(["dog and dog are friends"])
print("\nBoW for 'dog and dog are friends':", new_doc.toarray())

### 🔧 TODO 5.1
Rebuild the vectorizer with `binary=True` (so it only records *presence*, not *count*) and transform `"dog and dog are friends"` again. How does the output differ from above?

In [ ]:
# TODO: create count_vect_bin = CountVectorizer(binary=True), fit on `corpus`,
# then transform "dog and dog are friends" and print the result


### 🔧 TODO 5.2 — Bag of N-Grams
Create a `CountVectorizer` with `ngram_range=(1, 2)` (unigrams + bigrams), fit it on `corpus`, and print the vocabulary. How many features do you get compared to the plain Bag-of-Words above?

In [ ]:
# TODO: create a CountVectorizer with ngram_range=(1,2), fit_transform on `corpus`,
# and print count_vect_ngram.vocabulary_


## Part 6 — TF-IDF

`TfidfVectorizer` weighs words by how important they are to a document *relative to the whole corpus*.

In [ ]:
tfidf = TfidfVectorizer()
tfidf_rep = tfidf.fit_transform(corpus)

print("Vocabulary:", tfidf.get_feature_names_out())
print("IDF values:", tfidf.idf_)
print()
print("TF-IDF vector for D1 ('Dog bites man.'):")
print(tfidf_rep[0].toarray())

> **Note:** scikit-learn uses a slightly smoothed IDF formula, so these numbers won't exactly match the by-hand values from the lecture slides (dog=0.136, bites=0.17, etc.) — but the *relative* pattern (common words score lower, rare words score higher) will be the same.

### 🔧 TODO 6.1 — Cosine Similarity
Use `cosine_similarity` to compute the similarity between **all pairs** of documents in `corpus` using their TF-IDF vectors. Which two documents are most similar? Does that match your intuition from reading them?

In [ ]:
# TODO: compute the cosine similarity matrix for tfidf_rep and print it as a DataFrame
# hint: cosine_similarity(tfidf_rep) returns a 4x4 matrix

sim_matrix = None  # replace with your code
pd.DataFrame(sim_matrix, index=["D1","D2","D3","D4"], columns=["D1","D2","D3","D4"]) if sim_matrix is not None else None

**Your answer here (which documents are most similar, and why?):**

> 💡 *Hint: if two documents end up with a similarity of exactly 1.0, think back to the lecture — which text representations ignore word order? Does "Dog bites man" really mean the same thing as "Man bites dog"?*


## Part 7 — Word Embeddings with Word2Vec

Our toy corpus is far too small to learn meaningful embeddings (Word2Vec needs millions of words of context). So we'll do two things:

1. **Train a tiny Word2Vec model ourselves** on a slightly bigger sample corpus, just to see the training process.
2. **Load pre-trained embeddings** (trained on billions of words) to explore real analogies like `King - Man + Woman ≈ Queen`.

In [ ]:
!pip install gensim -q
import gensim
from gensim.models import Word2Vec

print("gensim version:", gensim.__version__)

In [ ]:
# A slightly bigger sample corpus (still tiny, just for demonstration)
training_sentences = [
    "the dog barked at the man".split(),
    "the man bit the dog".split(),
    "the dog ate the meat".split(),
    "the man ate the food".split(),
    "the cat chased the dog".split(),
    "the king ruled the land".split(),
    "the queen ruled the land".split(),
    "the man is a king".split(),
    "the woman is a queen".split(),
]

model = Word2Vec(sentences=training_sentences, vector_size=50, window=3, min_count=1, workers=2, epochs=200)
print("Vocabulary:", list(model.wv.index_to_key))

### 🔧 TODO 7.1
Use `model.wv.most_similar("dog")` to find the words our tiny model thinks are most similar to `"dog"`. Don't expect great results — this corpus is far too small! That's exactly the point: **Word2Vec needs a lot of data.**

In [ ]:
# TODO: print the most similar words to "dog" using model.wv.most_similar(...)


### Pre-trained embeddings: real analogies

Now let's load a small pre-trained embedding model (trained on Wikipedia + Gigaword, ~66MB) so we can try genuine analogies.

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")
print("Loaded", len(glove), "word vectors")

In [ ]:
# The classic analogy from the lecture: King - Man + Woman ≈ Queen
result = glove.most_similar(positive=["woman", "king"], negative=["man"], topn=5)
result

### 🔧 TODO 7.2
Try your **own analogy** using `glove.most_similar(positive=[...], negative=[...])`. Some ideas:
- `paris - france + italy ≈ ?`
- `bigger - big + small ≈ ?`

Then use `glove.most_similar("nlp")` (or another word of your choice) to see what words the model considers similar.

In [ ]:
# TODO: try your own analogy and a most_similar() lookup


## Part 8 — Wrap-Up

### 🔧 TODO 8.1 — Summary Table
Build a small `pandas` DataFrame comparing the text representation methods we've covered, with columns `Method`, `Handles OOV?`, `Captures Word Order?`, `Captures Word Similarity?`. Rows: One-Hot Encoding, Bag of Words, Bag of N-Grams, TF-IDF, Word2Vec.

In [ ]:
# TODO: build and display the comparison DataFrame
summary_rows = [
    # {"Method": "One-Hot Encoding", "Handles OOV?": "No", "Captures Word Order?": "No", "Captures Word Similarity?": "No"},
]

pd.DataFrame(summary_rows)

### Reflection

In 2–3 sentences: which text representation method from today would you choose for a **spam classifier**, and which would you choose for a **search engine that needs to understand synonyms**? Why?

**Your answer here:**

---
*End of lab. Save your notebook (File → Save a copy in Drive) before you leave.*